In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import RobustScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    precision_recall_curve, roc_curve, average_precision_score
)
import warnings
import shap 
warnings.filterwarnings('ignore')

f:\app space\Anaconda\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [8]:
# Load data
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')

# Handle missing values in TotalCharges
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'].fillna(df['MonthlyCharges'] * df['tenure'], inplace=True)

# Separate features and target
X = df.drop(['Churn', 'customerID'], axis=1)
y = df['Churn'].map({'Yes': 1, 'No': 0})

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")

Training set size: 5634
Test set size: 1409


In [10]:
import pandas as pd
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin


class TelcoFeatureEngineer(BaseEstimator, TransformerMixin):
    """
    Feature engineering transformer for Telco churn dataset.
    
    Creates:
    - Engagement and loyalty features (tenure groups, new/loyal customer flags)
    - Service usage features (add-ons, streaming, protection bundles)
    - Payment and contract risk features (auto-pay, high-risk combinations)
    - Derived financial features (AvgChargePerMonth, CLV, price hike flag)
    - Family and demographic risk features
    - Service type categorization (Both/PhoneOnly/InternetOnly/Neither)
    - Engagement score summarizing multiple behaviors
    """
    
    def __init__(
        self,
        new_customer_threshold=6,
        loyal_customer_threshold=24,
        price_hike_ratio=1.05,
        tenure_bins=None,
        tenure_labels=None,
    ):
        self.new_customer_threshold = new_customer_threshold
        self.loyal_customer_threshold = loyal_customer_threshold
        self.price_hike_ratio = price_hike_ratio
        
        # Default tenure grouping
        self.tenure_bins = tenure_bins or [0, 12, 24, 48, np.inf]
        self.tenure_labels = tenure_labels or ["0-12", "12-24", "24-48", "48+"]

        # Column groups for easier maintenance
        self._addon_cols = [
            "OnlineSecurity", "OnlineBackup", "DeviceProtection",
            "TechSupport", "StreamingTV", "StreamingMovies"
        ]
        self._protection_cols = ["OnlineSecurity", "DeviceProtection", "TechSupport"]
        self._streaming_cols = ["StreamingTV", "StreamingMovies"]

    def fit(self, X, y=None):
        # No fitting needed; just return self
        return self

    def transform(self, X):
        X = X.copy()
        
        # 1. Financial features
        X = self._add_financial_features(X)
        
        # 2. Contract & payment features
        X = self._add_contract_payment_features(X)
        
        # 3. Service usage & engagement features
        X = self._add_service_usage_features(X)
        
        # 4. Family & demographic risk features
        X = self._add_family_demographic_features(X)
        
        # 5. Tenure-based features
        X = self._add_tenure_features(X)
        
        # 6. Protection & streaming bundles
        X = self._add_protection_streaming_features(X)
        
        # 7. Service type categorization
        X = self._add_service_type_features(X)
        
        # 8. Overall engagement score
        X = self._add_engagement_score(X)
        
        return X

    # --------------------------
    # Helper methods (modular)
    # --------------------------
    
    def _add_financial_features(self, X):
        """Add AvgChargePerMonth, CLV, and price hike flag."""
        # Avoid division by zero for tenure
        tenure_safe = X["tenure"].replace(0, 1)
        
        X["AvgChargePerMonth"] = X["TotalCharges"] / tenure_safe
        X["CLV"] = X["MonthlyCharges"] * X["tenure"]
        
        # Price hike detection
        avg_charge_safe = X["AvgChargePerMonth"].replace(0, np.nan)
        ratio = X["MonthlyCharges"] / avg_charge_safe
        X["Charge_Increase_Ratio"] = ratio.fillna(1.0)
        X["Recent_Price_Hike"] = (
            X["Charge_Increase_Ratio"] > self.price_hike_ratio
        ).astype(int)
        
        return X
    
    def _add_contract_payment_features(self, X):
        """Add contract type, auto-pay, and high-risk payment flags."""
        X["IsMonthContract"] = (X["Contract"] == "Month-to-month").astype(int)
        
        # Auto-pay: any payment method with "automatic" in name
        X["AutoPay"] = X["PaymentMethod"].str.contains("automatic", case=False).astype(int)
        
        # Interaction: paperless billing + auto-pay
        X["Paperless_AutoPay_Interaction"] = (
            (X["PaperlessBilling"] == "Yes").astype(int) * X["AutoPay"]
        )
        
        # High-risk combination: Electronic check + Month-to-month
        X["High_Risk_Payment_Contract"] = (
            (X["PaymentMethod"] == "Electronic check") & 
            (X["Contract"] == "Month-to-month")
        ).astype(int)
        
        return X
    
    def _add_service_usage_features(self, X):
        """Add add-on counts, multiple lines, and charges per service."""
        # Number of add-on services
        X["NumAddOnServices"] = (X[self._addon_cols] == "Yes").sum(axis=1)
        
        # Multiple lines flag
        X["HasMultipleLines"] = (X["MultipleLines"] == "Yes").astype(int)
        
        # Charges per service (avoid division by zero)
        num_services_safe = X["NumAddOnServices"].replace(0, 1)
        X["ChargesPerService"] = X["MonthlyCharges"] / num_services_safe
        
        return X
    
    def _add_family_demographic_features(self, X):
        """Add family presence and senior citizen risk flags."""
        # Family presence: partner or dependents
        X["HasFamily"] = (
            (X["Partner"] == "Yes") | (X["Dependents"] == "Yes")
        ).astype(int)
        
        # Senior citizen without family support (higher risk)
        X["FamilyRisk"] = (
            (X["SeniorCitizen"] == 1) & (X["HasFamily"] == 0)
        ).astype(int)
        
        return X
    
    def _add_tenure_features(self, X):
        """Add tenure groups and new/loyal customer flags."""
        # Tenure groups
        X["TenureGroup"] = pd.cut(
            X["tenure"],
            bins=self.tenure_bins,
            labels=self.tenure_labels,
            right=False
        ).astype(str)
        
        # New vs loyal customer flags
        X["IsNewCustomer"] = (X["tenure"] <= self.new_customer_threshold).astype(int)
        X["IsLoyalCustomer"] = (X["tenure"] >= self.loyal_customer_threshold).astype(int)
        
        return X
    
    def _add_protection_streaming_features(self, X):
        """Add protection and streaming bundle flags."""
        # Protection features
        X["HasProtection"] = (X[self._protection_cols] == "Yes").any(axis=1).astype(int)
        X["FullProtectionBundle"] = (
            X[self._protection_cols] == "Yes"
        ).all(axis=1).astype(int)
        
        # Streaming usage
        X["UsesStreaming"] = (X[self._streaming_cols] == "Yes").any(axis=1).astype(int)
        
        # Fiber optic internet flag
        X["FiberUser"] = (X["InternetService"] == "Fiber optic").astype(int)
        
        return X
    
    def _add_service_type_features(self, X):
        """Categorize service type: Both/PhoneOnly/InternetOnly/Neither."""
        has_phone = X["PhoneService"] == "Yes"
        has_internet = X["InternetService"] != "No"
        
        X["Service_Type"] = np.select(
            [
                has_phone & has_internet,
                has_phone & ~has_internet,
                ~has_phone & has_internet
            ],
            ["Both", "PhoneOnly", "InternetOnly"],
            default="Neither"
        )
        
        return X
    
    def _add_engagement_score(self, X):
        """Create a composite engagement score."""
        X["EngagementScore"] = (
            X["NumAddOnServices"] +
            X["HasProtection"] +
            X["UsesStreaming"] +
            X["AutoPay"]
        )
        return X

In [11]:
BASE_NUMERIC_FEATURES = ['tenure', 'MonthlyCharges', 'TotalCharges']
BASE_BINARY_FEATURES = ['gender', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']
BASE_CATEGORICAL_FEATURES = [
    'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
    'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
    'Contract', 'PaymentMethod'
]

ENGINEERED_NUMERIC_FEATURES = [
    'AvgChargePerMonth', 'CLV', 'Charge_Increase_Ratio',
    'NumAddOnServices', 'ChargesPerService', 'EngagementScore'
]

ENGINEERED_BINARY_FEATURES = [
    'Recent_Price_Hike', 'IsMonthContract', 'AutoPay',
    'Paperless_AutoPay_Interaction', 'High_Risk_Payment_Contract',
    'HasMultipleLines', 'HasFamily', 'FamilyRisk',
    'IsNewCustomer', 'IsLoyalCustomer', 'HasProtection',
    'FullProtectionBundle', 'UsesStreaming', 'FiberUser'
]

ENGINEERED_CATEGORICAL_FEATURES = [
    'TenureGroup', 'Service_Type'
]

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', RobustScaler())
])

binary_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', drop='first'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('base_num', numeric_transformer, BASE_NUMERIC_FEATURES),
        ('eng_num', numeric_transformer, ENGINEERED_NUMERIC_FEATURES),
        ('base_binary', binary_transformer, BASE_BINARY_FEATURES),
        ('eng_binary', binary_transformer, ENGINEERED_BINARY_FEATURES),
        ('base_cat', categorical_transformer, BASE_CATEGORICAL_FEATURES),
        ('eng_cat', categorical_transformer, ENGINEERED_CATEGORICAL_FEATURES)
    ]
)


In [12]:
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import RFECV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

######################################################
# RFECV
######################################################

rfecv_model = Pipeline([
    ("feature_engineer", TelcoFeatureEngineer()),

    ("preprocessor", preprocessor),

    ("feature_selection",
        RFECV(
            estimator=LogisticRegression(
                random_state=42,
                max_iter=1000
            ),
            step=1,
            cv=5,
            scoring="roc_auc",
            n_jobs=-1
        )
    ),

    ("classifier",
        LogisticRegression(
            random_state=42,
            max_iter=1000
        )
    )
])

rfecv_model.fit(X_train, y_train)

rfecv_auc = roc_auc_score(
    y_test,
    rfecv_model.predict_proba(X_test)[:,1]
)

print(f"RFECV ROC-AUC : {rfecv_auc:.4f}")

RFECV ROC-AUC : 0.8458


In [13]:
feature_names = preprocessor.get_feature_names_out()

selector = rfecv_model.named_steps["feature_selection"]

selected_features_rfecv = feature_names[
    selector.get_support()
]

print("Number of selected features:",
      len(selected_features_rfecv))

print(selected_features_rfecv)

Number of selected features: 38
['base_num__tenure' 'eng_num__AvgChargePerMonth'
 'eng_num__NumAddOnServices' 'eng_num__EngagementScore'
 'base_binary__Dependents' 'base_binary__PhoneService'
 'base_binary__PaperlessBilling' 'eng_binary__Recent_Price_Hike'
 'eng_binary__IsMonthContract' 'eng_binary__AutoPay'
 'eng_binary__Paperless_AutoPay_Interaction'
 'eng_binary__HasMultipleLines' 'eng_binary__FamilyRisk'
 'eng_binary__IsNewCustomer' 'eng_binary__IsLoyalCustomer'
 'eng_binary__HasProtection' 'eng_binary__UsesStreaming'
 'eng_binary__FiberUser' 'base_cat__MultipleLines_Yes'
 'base_cat__InternetService_Fiber optic' 'base_cat__InternetService_No'
 'base_cat__OnlineSecurity_No internet service'
 'base_cat__OnlineSecurity_Yes'
 'base_cat__OnlineBackup_No internet service' 'base_cat__OnlineBackup_Yes'
 'base_cat__DeviceProtection_No internet service'
 'base_cat__TechSupport_No internet service' 'base_cat__TechSupport_Yes'
 'base_cat__StreamingTV_No internet service' 'base_cat__StreamingTV

In [14]:
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import RandomForestClassifier

######################################################
# Random Forest Feature Selection
######################################################

rf_selection_model = Pipeline([
    ("feature_engineer", TelcoFeatureEngineer()),

    ("preprocessor", preprocessor),

    ("feature_selection",
        SelectFromModel(
            estimator=RandomForestClassifier(
                n_estimators=300,
                random_state=42,
                n_jobs=-1
            ),
            threshold="median"
        )
    ),

    ("classifier",
        LogisticRegression(
            random_state=42,
            max_iter=1000
        )
    )
])

rf_selection_model.fit(X_train, y_train)

rf_auc = roc_auc_score(
    y_test,
    rf_selection_model.predict_proba(X_test)[:,1]
)

print(f"Tree Importance ROC-AUC : {rf_auc:.4f}")

Tree Importance ROC-AUC : 0.8402


In [15]:
feature_names = preprocessor.get_feature_names_out()

selector = rf_selection_model.named_steps["feature_selection"]

selected_features_rf = feature_names[
    selector.get_support()
]

print(len(selected_features_rf))
print(selected_features_rf)

27
['base_num__tenure' 'base_num__MonthlyCharges' 'base_num__TotalCharges'
 'eng_num__AvgChargePerMonth' 'eng_num__CLV'
 'eng_num__Charge_Increase_Ratio' 'eng_num__NumAddOnServices'
 'eng_num__ChargesPerService' 'eng_num__EngagementScore'
 'base_binary__gender' 'base_binary__Partner' 'base_binary__Dependents'
 'base_binary__PaperlessBilling' 'eng_binary__IsMonthContract'
 'eng_binary__High_Risk_Payment_Contract' 'eng_binary__HasFamily'
 'eng_binary__FamilyRisk' 'eng_binary__IsNewCustomer'
 'eng_binary__IsLoyalCustomer' 'eng_binary__HasProtection'
 'eng_binary__FiberUser' 'base_cat__InternetService_Fiber optic'
 'base_cat__OnlineSecurity_Yes' 'base_cat__OnlineBackup_Yes'
 'base_cat__TechSupport_Yes' 'base_cat__Contract_Two year'
 'base_cat__PaymentMethod_Electronic check']


In [16]:
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import mutual_info_classif

######################################################
# Mutual Information
######################################################

mi_selection_model = Pipeline([
    ("feature_engineer", TelcoFeatureEngineer()),

    ("preprocessor", preprocessor),

    ("feature_selection",
        SelectKBest(
            score_func=mutual_info_classif,
            k=20
        )
    ),

    ("classifier",
        LogisticRegression(
            random_state=42,
            max_iter=1000
        )
    )
])

mi_selection_model.fit(X_train, y_train)

mi_auc = roc_auc_score(
    y_test,
    mi_selection_model.predict_proba(X_test)[:,1]
)

print(f"Mutual Information ROC-AUC : {mi_auc:.4f}")

Mutual Information ROC-AUC : 0.8381


In [17]:
feature_names = preprocessor.get_feature_names_out()

selector = mi_selection_model.named_steps["feature_selection"]

selected_features_mi = feature_names[
    selector.get_support()
]

print(len(selected_features_mi))
print(selected_features_mi)

20
['base_num__tenure' 'base_num__MonthlyCharges' 'base_num__TotalCharges'
 'eng_num__CLV' 'eng_num__ChargesPerService' 'eng_num__EngagementScore'
 'eng_binary__IsMonthContract' 'eng_binary__High_Risk_Payment_Contract'
 'eng_binary__IsNewCustomer' 'eng_binary__IsLoyalCustomer'
 'eng_binary__FiberUser' 'base_cat__InternetService_Fiber optic'
 'base_cat__InternetService_No'
 'base_cat__OnlineBackup_No internet service'
 'base_cat__DeviceProtection_No internet service'
 'base_cat__TechSupport_No internet service'
 'base_cat__StreamingMovies_No internet service'
 'base_cat__Contract_Two year' 'base_cat__PaymentMethod_Electronic check'
 'eng_cat__TenureGroup_48+']


In [20]:
comparison = pd.DataFrame({
    "Method":[
        "RFECV",
        "Tree Importance",
        "Mutual Information"
    ],
    "ROC-AUC":[
        rfecv_auc,
        rf_auc,
        mi_auc
    ],
    "Selected Features":[
        len(selected_features_rfecv),
        len(selected_features_rf),
        len(selected_features_mi)
    ]
})

comparison.sort_values("ROC-AUC", ascending=False)

,Method,ROC-AUC,Selected Features
0,RFECV,0.845821,38
1,Tree Importance,0.840213,27
2,Mutual Information,0.838091,20


In [22]:
fe = TelcoFeatureEngineer()

X_train_fe = fe.fit_transform(X_train)
X_test_fe = fe.transform(X_test)

print(X_train_fe.shape)
print(X_test_fe.shape)

(5634, 41)
(1409, 41)


In [23]:
# fit preprocessing

X_train_processed = preprocessor.fit_transform(X_train_fe)
X_test_processed = preprocessor.transform(X_test_fe)


feature_names = preprocessor.get_feature_names_out()


X_train_processed = pd.DataFrame(
    X_train_processed,
    columns=feature_names,
    index=X_train.index
)


X_test_processed = pd.DataFrame(
    X_test_processed,
    columns=feature_names,
    index=X_test.index
)


print(X_train_processed.shape)

(5634, 54)


In [ ]:
from mlxtend.feature_selection import SequentialFeatureSelector as SFS
from sklearn.ensemble import RandomForestClassifier


rf_selector = RandomForestClassifier(
    n_estimators=300,
    max_depth=6,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)


sfs = SFS(
    estimator=rf_selector,

    # تعداد ویژگی‌های نهایی
    k_features=(20,30),

    forward=True,

    # Floating=False یعنی فقط اضافه کردن
    floating=False,

    scoring="roc_auc",

    cv=5,

    n_jobs=-1,

    verbose=2
)


sfs.fit(
    X_train_processed,
    y_train
)

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 16 concurrent workers.
[Parallel(n_jobs=-1)]: Done   9 tasks      | elapsed:   17.6s
[Parallel(n_jobs=-1)]: Done  51 out of  54 | elapsed:   31.5s remaining:    1.8s
[Parallel(n_jobs=-1)]: Done  54 out of  54 | elapsed:   31.6s finished

[2026-07-18 14:03:30] Features: 1/30 -- score: 0.7464792751605038[Parallel(n_jobs=-1)]: Using backend LokyBackend with 16 concurrent workers.
[Parallel(n_jobs=-1)]: Done   9 tasks      | elapsed:    6.5s
[Parallel(n_jobs=-1)]: Done  49 out of  53 | elapsed:   23.7s remaining:    1.8s
[Parallel(n_jobs=-1)]: Done  53 out of  53 | elapsed:   23.9s finished

[2026-07-18 14:03:54] Features: 2/30 -- score: 0.8034523930023869[Parallel(n_jobs=-1)]: Using backend LokyBackend with 16 concurrent workers.
[Parallel(n_jobs=-1)]: Done   9 tasks      | elapsed:    6.6s
[Parallel(n_jobs=-1)]: Done  48 out of  52 | elapsed:   20.9s remaining:    1.7s
[Parallel(n_jobs=-1)]: Done  52 out of  52 | elapsed:   23.1s fini

In [ ]:
sfs_results = pd.DataFrame.from_dict(
    sfs.get_metric_dict()
).T


sfs_results